In [ ]:
from google.adk.models import Gemini
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner
from dotenv import load_dotenv

In [6]:
load_dotenv()

True

In [ ]:
ORCHESTRATOR_SYSTEM_PROMPT = """
You are the Orchestrator Agent of Skillix — a multi-agent personalized tutoring system.

Your job is to manage the entire learning journey by deciding which phase to activate based on the current state.

### Phase Detection Rules (You decide based on session state and user input):

1. **New Session (no user_profile in state)**  
   → Collect preferences ONCE using friendly conversation  
   → Confirm and output exactly:
   <CONFIRMED_PROFILE>
   {
     "topic": "...",
     "level": "Beginner|Intermediate|Advanced",
     "style": "theory-first|application-first|hybrid",
     "mode": "interactive|guided"
   }
   </CONFIRMED_PROFILE>

2. **Research Phase** (has user_profile but no syllabus in state)  
   → Automatically trigger research workflow (no user interaction needed)
   → Respond: "I'm researching the best resources and creating your personalized syllabus..."

3. **Teaching Phase** (has syllabus in state, ongoing conversation)  
   → Delegate EVERY user message to the Teaching + Evaluating loop
   → Just forward — do not interfere

4. **End Session** (user says "finish", "done", "generate report", etc.)  
   → Trigger final report generation
   → Respond: "Generating your learning report..."

### Tools You Have:
- research_workflow: Runs Context Gatherer → Planner sequentially
- teaching_loop: Runs Teaching ↔ Evaluating in a loop until done
- generate_report: Runs the Final Report agent

### Response Rules:
- For phase 1: Only collect and confirm profile
- For phase 2–4: Short status message + tool call
- NEVER teach content yourself
- Be warm, encouraging, and professional
"""

In [7]:
orchestrator_agent = Agent(
    name="OrchestratorAgent",
    model=Gemini(model="gemini-2.5-flash"),
    instruction=ORCHESTRATOR_SYSTEM_PROMPT
)

In [10]:
runner = InMemoryRunner(agent=orchestrator_agent)

In [11]:
response = await runner.run_debug("I need to learn Google's Agent Development Kit and I am a beginner and I like theory-first and interactive mode")

print(response)


 ### Created new session: debug_session_id

User > I need to learn Google's Agent Development Kit and I am a beginner and I like theory-first and interactive mode
OrchestratorAgent > That sounds like an exciting topic! Let's get started.

<CONFIRMED_PROFILE>
{
  "topic": "Google's Agent Development Kit",
  "level": "Beginner",
  "style": "theory-first",
  "mode": "interactive"
}
</CONFIRMED_PROFILE>
[Event(model_version='gemini-2.5-flash', content=Content(
  parts=[
    Part(
      text="""That sounds like an exciting topic! Let's get started.

<CONFIRMED_PROFILE>
{
  "topic": "Google's Agent Development Kit",
  "level": "Beginner",
  "style": "theory-first",
  "mode": "interactive"
}
</CONFIRMED_PROFILE>"""
    ),
  ],
  role='model'
), grounding_metadata=None, partial=None, turn_complete=None, finish_reason=<FinishReason.STOP: 'STOP'>, error_code=None, error_message=None, interrupted=None, custom_metadata=None, usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_

The above `<CONFIRMED_PROFILE>` needs to be stored in the memory for accessing other agents and this orchestrator agent needs to start the research phase (sequential agents Context Gathering and Planning agents)